In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=73d0c40d06259f0f9c227da8653e4df4356ea6dcdcbf98271a272cb2e65169b7
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.
simulator = BasicSimulator()

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)              # |0> -> |+> = (1/sqrt2)(|0> + |1>)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    # Get the measured bit as an integer
    return int(list(result.get_counts().keys())[0])

def quantum_random_bits(n):
    """Generate n random bits."""
    return [quantum_random_bit() for _ in range(n)]

def quantum_random_basis(n):
    """Generate n random basis choices: 's' (standard) or 'd' (diagonal)."""
    return ['s' if quantum_random_bit() == 0 else 'd' for _ in range(n)]


In [3]:
def alice_encode(bit, basis):
    """
    Alice prepares one qubit according to her bit and basis choice.
    Standard basis: 0 -> |0>, 1 -> |1>
    Diagonal basis: 0 -> |+>, 1 -> |->
    Returns a QuantumCircuit holding the prepared qubit (no measurement yet).
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # flip |0> to |1> first
    if basis == 'd':
        qc.h(0)          # rotate into diagonal basis: |0>->|+>, |1>->|->
    return qc

def bob_measure(qc, basis):
    """
    Bob measures the qubit in his chosen basis.
    For diagonal-basis measurement: apply H first, then standard measure
    (this is the trick from Lecture 3b slide 15).
    """
    if basis == 'd':
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    return int(list(result.get_counts().keys())[0])


def bb84_no_attacker(n_qubits=40, sample_fraction=0.5, threshold=0.0):
    """
    Run BB84 with no eavesdropper.
    n_qubits: how many qubits Alice sends initially
    sample_fraction: fraction of the matching-basis bits to sacrifice for tamper-check
    threshold: max allowed mismatch rate in the sample before declaring an attack
    """
    print("="*60)
    print("BB84 WITHOUT ATTACKER")
    print("="*60)

    # --- ALICE'S SIDE ---
    alice_bits   = quantum_random_bits(n_qubits)
    alice_bases  = quantum_random_basis(n_qubits)

    # --- BOB'S SIDE: pick his bases ahead of time ---
    bob_bases    = quantum_random_basis(n_qubits)

    # --- QUANTUM CHANNEL: Alice prepares, Bob measures ---
    bob_bits = []
    for i in range(n_qubits):
        qc = alice_encode(alice_bits[i], alice_bases[i])   # Alice
        bit = bob_measure(qc, bob_bases[i])                # Bob
        bob_bits.append(bit)

    # --- PUBLIC DISCUSSION: keep only matching-basis positions ---
    matching_positions = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_key_raw = [alice_bits[i] for i in matching_positions]
    bob_key_raw   = [bob_bits[i]   for i in matching_positions]

    # --- TAMPER CHECK: publicly compare a random subset ---
    n_sample = max(1, int(len(matching_positions) * sample_fraction))
    # pick which positions of the matching list to sacrifice (quantum-random)
    sample_idx = sorted(set(quantum_random_bit() * (len(matching_positions)-1) // 1
                            for _ in range(n_sample)))  # simple sampler
    # safer: just take the first n_sample (their order is already random anyway)
    sample_idx = list(range(n_sample))

    mismatches = sum(1 for i in sample_idx if alice_key_raw[i] != bob_key_raw[i])
    mismatch_rate = mismatches / n_sample

    # remaining bits become the final key
    final_alice_key = [alice_key_raw[i] for i in range(len(alice_key_raw)) if i not in sample_idx]
    final_bob_key   = [bob_key_raw[i]   for i in range(len(bob_key_raw))   if i not in sample_idx]

    # --- REPORT ---
    print(f"Qubits sent:            {n_qubits}")
    print(f"Matching-basis bits:    {len(matching_positions)}  (~half, as expected)")
    print(f"Sacrificed for check:   {n_sample}")
    print(f"Mismatches in sample:   {mismatches}  ({mismatch_rate:.1%})")
    print(f"Threshold for alarm:    {threshold:.1%}")
    if mismatch_rate > threshold:
        print(">>> ATTACK SUSPECTED — discard the key!")
    else:
        print(">>> Channel looks clean — keep the key.")
    print(f"Final key length:       {len(final_alice_key)}")
    print(f"Alice's final key:      {final_alice_key}")
    print(f"Bob's final key:        {final_bob_key}")
    print(f"Keys agree:             {final_alice_key == final_bob_key}")

bb84_no_attacker(n_qubits=40)

BB84 WITHOUT ATTACKER
Qubits sent:            40
Matching-basis bits:    14  (~half, as expected)
Sacrificed for check:   7
Mismatches in sample:   0  (0.0%)
Threshold for alarm:    0.0%
>>> Channel looks clean — keep the key.
Final key length:       7
Alice's final key:      [0, 1, 1, 1, 1, 1, 0]
Bob's final key:        [0, 1, 1, 1, 1, 1, 0]
Keys agree:             True
